# Experiment: Case 001 Minimum Hematologist Packet Gate

Objective: compress the case-001 genotype, immune/transfusion,
iron/chelation, organ-risk, and referral gates into one doctor-ready minimum
record packet.

Success criteria:
- no diagnosis or treatment advice;
- no raw medical records or identifiers;
- one ranked list that explains what to bring or request first;
- every packet area maps back to an existing source-backed gate.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Final

CASE_ID: Final = "case-001"
CURRENT_LABELS: Final = {
    "genotype": "phenotype_only",
    "immune_transfusion": "immune_transfusion_packet_missing",
    "iron": "iron_packet_missing",
    "trial_referral": "trial_referral_not_ready",
}

CURRENT_LABELS

## Scoring Model

The score is not a medical severity score. It ranks missing records by how much
they block a safe clinician conversation and research routing.


In [ ]:
@dataclass(frozen=True)
class PacketArea:
    """A case-001 record area needed before patient-specific interpretation."""

    name: str
    label: str
    request: str
    blocks_diagnosis: int
    blocks_transfusion_explanation: int
    blocks_safety: int
    blocks_referral: int
    source_gate: str

    def score(self) -> int:
        """Rank record areas by how much they block current decisions."""
        return (
            self.blocks_diagnosis
            + self.blocks_transfusion_explanation
            + self.blocks_safety
            + self.blocks_referral
        )


areas = [
    PacketArea(
        name="current_subtype_and_genotype",
        label=CURRENT_LABELS["genotype"],
        request=(
            "Current diagnosis note, TDT/NTDT/HbE-beta/mixed classification, "
            "HPLC fractions, HBB, HBA1/HBA2, and HPFH/HBG modifier context."
        ),
        blocks_diagnosis=5,
        blocks_transfusion_explanation=4,
        blocks_safety=3,
        blocks_referral=5,
        source_gate="case001-high-hbf-genotype-evidence-gate",
    ),
    PacketArea(
        name="transfusion_burden_and_response",
        label="transfusion_dependent_burden_unquantified",
        request=(
            "Dates, pre-transfusion Hb, units, volume, body weight, reactions, "
            "and annual ml/kg/year or pure red-cell volume if available."
        ),
        blocks_diagnosis=3,
        blocks_transfusion_explanation=5,
        blocks_safety=4,
        blocks_referral=5,
        source_gate="weekly-transfusion-differential-map",
    ),
    PacketArea(
        name="immune_and_blood_bank_packet",
        label=CURRENT_LABELS["immune_transfusion"],
        request=(
            "Antibody screen, named alloantibodies, DAT/direct Coombs, "
            "matching policy, red-cell phenotype/genotype, and reaction history."
        ),
        blocks_diagnosis=2,
        blocks_transfusion_explanation=5,
        blocks_safety=5,
        blocks_referral=5,
        source_gate="case001-immune-transfusion-record-gate",
    ),
    PacketArea(
        name="iron_chelation_organ_risk_packet",
        label=CURRENT_LABELS["iron"],
        request=(
            "Ferritin trend, LIC, cardiac T2*, chelator identity and plan, "
            "toxicity monitoring, endocrine/bone screen, and ferritin context."
        ),
        blocks_diagnosis=2,
        blocks_transfusion_explanation=3,
        blocks_safety=5,
        blocks_referral=5,
        source_gate="case001-iron-chelation-organ-risk-record-gate",
    ),
    PacketArea(
        name="autoimmune_and_spleen_context",
        label="autoimmune_context_unknown",
        request=(
            "Exact autoimmune diagnosis, tests, medicines, spleen size/status, "
            "hemolysis markers, infection context, and specialist notes."
        ),
        blocks_diagnosis=2,
        blocks_transfusion_explanation=4,
        blocks_safety=5,
        blocks_referral=4,
        source_gate="case001-research-routing-matrix",
    ),
    PacketArea(
        name="advanced_therapy_referral_packet",
        label=CURRENT_LABELS["trial_referral"],
        request=(
            "Prior luspatercept, mitapivat, HSCT, gene therapy, CRISPR, "
            "trial, referral, access, fertility, and infection-status notes."
        ),
        blocks_diagnosis=2,
        blocks_transfusion_explanation=2,
        blocks_safety=4,
        blocks_referral=5,
        source_gate="trial-referral-no-lab-gate",
    ),
]

ranked_areas = sorted(areas, key=lambda area: (-area.score(), area.name))
[(area.name, area.label, area.score()) for area in ranked_areas]

## Minimum Packet Output

The output is a compact checklist. It deliberately asks for records and labels,
not for treatment changes.


In [ ]:
minimum_packet = {
    "case_id": CASE_ID,
    "status": "not_treatment_advice",
    "active_labels": CURRENT_LABELS,
    "top_three": [area.request for area in ranked_areas[:3]],
    "remaining": [area.request for area in ranked_areas[3:]],
    "source_gates": [area.source_gate for area in ranked_areas],
    "boundary": (
        "Patient-specific candidate ranking is blocked until the packet is "
        "reviewed by a hematologist or marked unavailable."
    ),
}

minimum_packet

## Interpretation

The minimum packet has three first asks:

1. current subtype/genotype/Hb-fraction packet;
2. transfusion burden and response packet;
3. immune and blood-bank packet.

The iron/chelation organ-risk packet remains close behind because it controls
safety and referral interpretation. This ranking does not mean iron is less
important; it means diagnosis and transfusion mechanics must be known before the
rest of the case can be routed cleanly.
